In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
# LangSmith Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANG_SMITH_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3.6-27b", groq_api_key = os.environ["GROQ_API_KEY"])
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002A1576AF0E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A1576AFB60>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=llm.invoke(messages)

In [7]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input: "Hello How are you?"\n   - Language: English\n   - Task: Translate to French\n\n2.  **Identify Key Components:**\n   - "Hello" -> Greeting\n   - "How are you?" -> Question about well-being\n\n3.  **Determine French Equivalents:**\n   - "Hello" -> "Bonjour" (formal/standard), "Salut" (informal)\n   - "How are you?" -> "Comment allez-vous ?" (formal/plural), "Comment vas-tu ?" (informal/singular), "Ça va ?" (casual)\n\n4.  **Consider Context/Register:**\n   - Since no context is provided, it\'s best to provide the standard/most common translation, possibly noting formality levels if needed.\n   - Standard/neutral: "Bonjour, comment allez-vous ?" or "Bonjour, comment vas-tu ?"\n   - I\'ll go with the most universally appropriate: "Bonjour, comment allez-vous ?" (formal/polite) or "Bonjour, comment vas-tu ?" (informal). Actually, in French, it\'s common to just say "Bonjour, comment ça va ?" for a neutral/c

In [9]:
# Using LCEL - Chain the components

chain = llm|parser
chain.invoke(messages)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Source text: "Hello How are you?"\n   - Target language: French\n   - Task: Translation\n\n2.  **Identify Key Components:**\n   - "Hello" -> Greeting\n   - "How are you?" -> Question about well-being\n   - Note: There\'s a missing space/punctuation between "Hello" and "How", but it\'s clearly meant to be "Hello. How are you?" or "Hello, how are you?"\n\n3.  **Determine French Equivalents:**\n   - "Hello" -> "Bonjour" (standard, polite) or "Salut" (informal)\n   - "How are you?" -> "Comment allez-vous ?" (formal/plural) or "Comment vas-tu ?" (informal) or "Comment ça va ?" (common, neutral/informal)\n   - Combine them naturally in French: "Bonjour, comment allez-vous ?" or "Bonjour, comment ça va ?"\n\n4.  **Consider Context/Register:**\n   - Since no context is provided, it\'s best to provide a standard/polite version and optionally note informal alternatives.\n   - Standard: "Bonjour, comment allez-vous ?"\n 

In [10]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following into {language}:"

prompt = ChatPromptTemplate.from_messages(
    [("system", generic_template), ("user", "{text}")]
)

In [11]:
prompt.invoke({"language": "Urdu", "text":"How are you"})

ChatPromptValue(messages=[SystemMessage(content='Translate the following into Urdu:', additional_kwargs={}, response_metadata={}), HumanMessage(content='How are you', additional_kwargs={}, response_metadata={})])

In [12]:
chain= prompt|llm|parser
chain.invoke({"language": "Urdu", "text":"How are you"})

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Input text:** "How are you"\n   - **Target language:** Urdu\n   - **Task:** Translation\n\n2.  **Identify Key Components:**\n   - The phrase is a common English greeting/question.\n   - It\'s informal/polite depending on context, but generally neutral.\n   - In Urdu, the equivalent common phrase is "آپ کیسے ہیں؟" (formal/polite) or "تو کیسے ہے؟" (informal).\n   - Since no context is provided, the standard/polite form is safest and most commonly expected.\n\n3.  **Determine Urdu Translation:**\n   - "How are you?" → آپ کیسے ہیں؟ (Aap kaise hain?)\n   - Alternative: آپ کا حال کیا ہے؟ (Aap ka haal kya hai?) - less common for direct greeting\n   - I\'ll go with the standard: آپ کیسے ہیں؟\n\n4.  **Verify Accuracy:**\n   - "آپ" = you (formal/polite)\n   - "کیسے" = how\n   - "ہیں" = are\n   - Matches perfectly.\n   - Roman Urdu (optional but helpful): Aap kaise hain?\n   - I\'ll provide the Urdu script as primary, 